# Глава 2. Данные и признаки

Код к главе 2 книги 1: разведочный анализ, пропуски, прямое кодирование, стандартизация,
взаимодействие признаков и конвейер scikit-learn (листинг 2.1<!-- lst:Конвейер -->). Сквозной пример —
синтетические данные о квартирах; цены условные.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markpolyak/ml-textbook-code/blob/main/book1/ch02_data.ipynb)

In [ ]:
# В Colab ставим пакет mlbook с данными книги (локально: pip install -e . в корне репозитория)
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/markpolyak/ml-textbook-code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mlbook import examples, flats

## Фрагмент выборки: шесть квартир

Табл. 2.1<!-- tbl:flats -->. Для квартиры № 4 год постройки неизвестен (`NaN`).

In [ ]:
small = examples.flats6()
small

**Пропуск.** Медиана известных годов — 2005; ею заполняем пропуск.

In [ ]:
print(small["год"].median())
small["год"] = small["год"].fillna(small["год"].median())

**Прямое кодирование.** Район превращается в индикаторы $f_Б$ и $f_В$; район А — базовый
(его столбец удалён, иначе сумма индикаторов совпала бы с фиктивным признаком $f_0 = 1$).
Получается матрица $\mathbf{F}$ из задачи 1.

In [ ]:
F = pd.get_dummies(small[["площадь", "район", "этаж", "год"]], columns=["район"],
                   drop_first=True, dtype=int)
F.insert(0, "f0", 1)
F = F[["f0", "площадь", "район_Б", "район_В", "этаж", "год"]]
F

**Стандартизация** (задача 2): $x' = (x - m)/\sigma$, где $\sigma$ — стандартное отклонение
с делителем $\ell$ (в NumPy так по умолчанию, в pandas — нет: там `ddof=1`).

In [ ]:
area = small["площадь"].to_numpy(dtype=float)
m, s = area.mean(), area.std()
print(m, round(s, 2))
print(((area - m) / s).round(2))

## Разведочный анализ

Полная синтетическая выборка — 300 квартир. Сначала — типы, диапазоны и пропуски.

In [ ]:
df = flats.load()
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
df.isna().mean().round(3)        # доля пропусков по столбцам

Распределение цены несимметрично, распределение её логарифма — почти симметрично (рис. 2.1<!-- fig:data-distribution -->).

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3))
a.hist(df["цена"], bins=25, edgecolor="white")
a.set_title("а) цена")
a.set_xlabel("цена, млн руб.")
a.set_ylabel("число квартир")
b.hist(np.log(df["цена"]), bins=25, color="C1", edgecolor="white")
b.set_title("б) логарифм цены")
b.set_xlabel("ln(цена)")
plt.show()

**Коэффициент корреляции Пирсона** — по формуле из разд. 2.2<!-- sec:Разведочный анализ --> и функцией NumPy.

In [ ]:
u, v = df["площадь"].to_numpy(), df["цена"].to_numpy()
du, dv = u - u.mean(), v - v.mean()
r = np.sum(du * dv) / np.sqrt(np.sum(du ** 2) * np.sum(dv ** 2))
print(round(r, 3), round(np.corrcoef(u, v)[0, 1], 3))

In [ ]:
df.select_dtypes("number").corr().round(2)

Цена связана с площадью и расстоянием до метро; этаж и год постройки на цену в этих
данных не влияют.

## Масштабирование

Площадь, этаж и расстояние до метро на общей оси до и после стандартизации (рис. 2.2<!-- fig:scaling -->).

In [ ]:
cols = ["площадь", "этаж", "метро"]
fig, axes = plt.subplots(2, 1, figsize=(8, 4))
rng = np.random.default_rng(3)
for ax, std in zip(axes, (False, True)):
    for k, c in enumerate(cols):
        x = df[c].to_numpy()
        if std:
            x = (x - x.mean()) / x.std()
        ax.scatter(x, k + rng.uniform(-0.2, 0.2, x.size), s=6, alpha=0.6)
    ax.set_yticks(range(3), cols)
    ax.invert_yaxis()
axes[0].set_title("а) исходные значения")
axes[1].set_title("б) после стандартизации")
plt.tight_layout()
plt.show()

## Взаимодействие признаков

Без взаимодействия прямые «цена от площади» в трёх районах параллельны; с признаками
«площадь × индикатор района» у каждого района свой наклон — цена квадратного метра (рис. 2.3<!-- fig:interaction -->).

In [ ]:
d = flats.make()
area, price, dist = d["area"], d["price"], d["district"]
ind = np.column_stack([(dist == k).astype(float) for k in flats.DISTRICTS])

F1 = np.column_stack([np.ones_like(area), ind[:, 1:], area])               # общий наклон
F2 = np.column_stack([np.ones_like(area), ind[:, 1:], area[:, None] * ind])  # свой наклон
t1 = np.linalg.lstsq(F1, price, rcond=None)[0]
t2 = np.linalg.lstsq(F2, price, rcond=None)[0]
print("общий наклон, млн руб./м²:", t1[3].round(3))
print("наклоны по районам А, Б, В:", t2[3:].round(3), " (заложено в данных:",
      list(flats.PRICE_M2.values()), "при метро = 0)")

In [ ]:
xs = np.linspace(20, 130, 2)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for k, name in enumerate(flats.DISTRICTS):
    m = dist == name
    for ax in axes:
        ax.scatter(area[m], price[m], s=6, alpha=0.5, color=f"C{k}")
    shift1 = 0 if k == 0 else t1[k]
    shift2 = 0 if k == 0 else t2[k]
    axes[0].plot(xs, t1[0] + shift1 + t1[3] * xs, color=f"C{k}", label=f"район {name}")
    axes[1].plot(xs, t2[0] + shift2 + t2[3 + k] * xs, color=f"C{k}")
axes[0].set_title("а) без взаимодействия")
axes[1].set_title("б) с взаимодействием")
for ax in axes:
    ax.set_xlabel("площадь, м²")
axes[0].set_ylabel("цена, млн руб.")
axes[0].legend()
plt.show()

## Конвейер: предобработка без утечки

Разобьём выборку на обучающую и тестовую части. Медиана для пропусков, средние и
стандартные отклонения для стандартизации и список районов вычисляются только по
обучающей части — это обеспечивает конвейер (листинг 2.1<!-- lst:Конвейер -->).

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["площадь", "этаж", "год", "район"]]
y = df["цена"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

num = make_pipeline(SimpleImputer(strategy="median"),
                    StandardScaler())
prep = make_column_transformer(
    (num, ["площадь", "этаж", "год"]),
    (OneHotEncoder(drop="first"), ["район"]))
model = make_pipeline(prep, LinearRegression())
# статистики вычисляются только по обучающей части
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

Что запомнил конвейер: медианы и средние — по обучающей части, веса модели — по
стандартизованным признакам и индикаторам районов Б и В.

In [ ]:
imputer = model[0].named_transformers_["pipeline"][0]
print("медианы:", imputer.statistics_)
print("признаки:", model[0].get_feature_names_out())
print("веса:", model[-1].coef_.round(3))

In [ ]:
from sklearn.metrics import mean_absolute_error

print("MAE на тесте, млн руб.:", round(mean_absolute_error(y_test, y_pred), 2))

**Новый признак.** Добавим расстояние до метро. Ошибка уменьшится, но немного: в этих данных
метро влияет на цену слабее, чем площадь и район (сравните корреляции выше).

In [ ]:
cols = ["площадь", "этаж", "год", "метро"]
X_train, X_test, y_train, y_test = train_test_split(
    df[cols + ["район"]], y, test_size=0.25, random_state=0)
model = make_pipeline(
    make_column_transformer((num, cols), (OneHotEncoder(drop="first"), ["район"])),
    LinearRegression())
model.fit(X_train, y_train)
print("MAE на тесте, млн руб.:", round(mean_absolute_error(y_test, model.predict(X_test)), 2))